In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from src.utils.preprocessing import wrangle_data
from src.ingestion.loaders import load_file
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve
from src.models.evaluate import evaluate, evaluate_anomaly, identify_best_model, print_final_results, find_best_threshold

In [ ]:
final_dataframe = wrangle_data(True)

In [ ]:
final_dataframe.head(4)

In [ ]:
final_dataframe.info()

In [ ]:
# prepare features and target
#TIMESTAMP is now beiing drop in wrangle function
X = final_dataframe.drop(columns=["IS_FRAUD"])
y = final_dataframe["IS_FRAUD"]

In [ ]:
cutoff = int(len(X) * 0.8)
X_train_full, y_train_full = X.iloc[: cutoff], y.iloc[:cutoff]
X_test, y_test =  X.iloc[cutoff: ], y.iloc[cutoff:]

cutoff_train_val = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full.iloc[:cutoff_train_val], y_train_full.iloc[:cutoff_train_val]
X_validation, y_validation = X_train_full.iloc[cutoff_train_val:], y_train_full.iloc[cutoff_train_val:]


In [ ]:
#X = 1048575
print("X length " + str(len(X)))
print("X_train length " +str(len(X_train)) + ", y_train length " +str(len(y_train)))
print("X_val length " +str(len(X_validation)) + ", y_val length " +str(len(y_validation)))
print("X_test length " +str(len(X_test))  + ", y_test length " +str(len(y_test)))

In [ ]:
#fraud rate from train dataset
fraud_rate = y_train.value_counts(normalize=True).iloc[1]
print(fraud_rate)

In [ ]:
logistic_regression_pipeline = Pipeline([
        ("smote",  SMOTE(random_state=42)),
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(max_iter=1000, random_state=42)),
    ])
logistic_regression_pipeline.fit(X_train, y_train)

In [ ]:
random_forest_pipeline = Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
random_forest_pipeline.fit(X_train, y_train)

In [ ]:
xgboost_pipeline =   Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=-1))])
xgboost_pipeline.fit(X_train, y_train)

In [ ]:
CONTAMINATION = round(fraud_rate,4)
isolation_forest = IsolationForest(contamination=CONTAMINATION, random_state=42, n_jobs=-1,)
isolation_forest.fit(X_train)

In [ ]:
results=[]
results.append(evaluate(logistic_regression_pipeline, X_validation, y_validation, "Logistic Regression"))
results.append(evaluate(random_forest_pipeline, X_validation, y_validation, "Random Forest"))
results.append(evaluate(xgboost_pipeline, X_validation, y_validation, "XGBoost"))
results.append(evaluate_anomaly(isolation_forest, X_validation, y_validation, "Isolation Forest"))


In [ ]:
overall_best_classifier_model_name = identify_best_model(results)

In [ ]:
# Tune the classification threshold on the VALIDATION set for whichever model
# identify_best_model picked, instead of relying on sklearn's default 0.5 cutoff.
_pipelines_by_name = {
    "Logistic Regression": logistic_regression_pipeline,
    "Random Forest": random_forest_pipeline,
    "XGBoost": xgboost_pipeline,
}

if overall_best_classifier_model_name in _pipelines_by_name:
    best_pipeline = _pipelines_by_name[overall_best_classifier_model_name]
    val_probs = best_pipeline.predict_proba(X_validation)[:, 1]

    # beta=1 -> plain F1. Try beta=2 to weight recall higher if missing fraud
    # is costlier than a false alarm for this use case.
    threshold_result = find_best_threshold(y_validation, val_probs, beta=1.0)
    best_threshold = threshold_result["threshold"]

    print(f"Best model    : {overall_best_classifier_model_name}")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"  F{threshold_result['beta']:g}-score : {threshold_result['f_score']:.4f}")
    print(f"  Precision  : {threshold_result['precision']:.4f}")
    print(f"  Recall     : {threshold_result['recall']:.4f}")
else:
    # e.g. Isolation Forest won - it has no predict_proba, so this F-beta
    # threshold sweep doesn't apply to it as-is.
    best_pipeline, best_threshold = None, None
    print(f"{overall_best_classifier_model_name} has no predict_proba; skipping threshold tuning.")

In [ ]:
# Just to check if Random Forest in the hybrid model notebook yield same evaluation result to ensure reproducibility
evaluate(random_forest_pipeline, X_test, y_test, "Random Forest", threshold=best_threshold)